Historical environmental dataset

In [ ]:
# Historical environmental dataset
historical_df = master_df[[
    "year",
    "avg_temperature",
    "tree_cover_loss_ha",
    "mangrove_area_ha"
]].copy()

historical_df.columns = [
    "year",
    "temperature",
    "tree_cover_loss_ha",
    "mangrove_area_ha"
]

historical_df["data_type"] = "Historical"


# Future environmental dataset
future_base_df = forecast_temp.merge(
    forecast_forest,
    on="year"
).merge(
    forecast_mangrove,
    on="year"
)

future_base_df = future_base_df[[
    "year",
    "predicted_temperature",
    "predicted_tree_cover_loss_ha",
    "predicted_mangrove_area_ha"
]].copy()

future_base_df.columns = [
    "year",
    "temperature",
    "tree_cover_loss_ha",
    "mangrove_area_ha"
]

future_base_df["data_type"] = "Forecast"


# Combine historical and future
combined_risk_df = pd.concat(
    [historical_df, future_base_df],
    ignore_index=True
)

combined_risk_df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

combined_risk_df[[
    "temp_score",
    "forest_loss_score",
    "mangrove_score"
]] = scaler.fit_transform(
    combined_risk_df[[
        "temperature",
        "tree_cover_loss_ha",
        "mangrove_area_ha"
    ]]
) * 100

In [ ]:
combined_risk_df["risk_score"] = (
    0.45 * combined_risk_df["temp_score"]
    +
    0.40 * combined_risk_df["forest_loss_score"]
    +
    0.15 * (100 - combined_risk_df["mangrove_score"])
)

In [ ]:
def classify_risk(score):
    if score <= 25:
        return "Low"
    elif score <= 50:
        return "Moderate"
    elif score <= 75:
        return "High"
    else:
        return "Critical"

combined_risk_df["risk_category"] = combined_risk_df["risk_score"].apply(classify_risk)

In [ ]:
future_risk_fixed = combined_risk_df[
    combined_risk_df["data_type"] == "Forecast"
].copy()

future_risk_fixed[[
    "year",
    "temperature",
    "tree_cover_loss_ha",
    "mangrove_area_ha",
    "risk_score",
    "risk_category"
]]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.plot(
    combined_risk_df["year"],
    combined_risk_df["risk_score"],
    marker="o",
    label="Risk Score"
)

plt.axvline(
    x=2024,
    linestyle="--",
    label="Forecast Starts"
)

plt.title("Karachi Environmental Risk Index (2000-2035)")
plt.xlabel("Year")
plt.ylabel("Risk Score")
plt.legend()
plt.grid(True)
plt.show()